In [2]:
"""
Leaf Scan: Vision & Logic Integration
This notebook demonstrates the training of a Convolutional Neural Network to identify 38 distinct classes of plant diseases, which will later be integrated with a JSON-based Knowledge Base for treatment recommendations.
"""

'\nPlant Disease Expert System: Vision & Logic Integration\nThis notebook demonstrates the training of a Convolutional Neural Network to identify 38 distinct classes of plant diseases, which will later be integrated with a JSON-based Knowledge Base for treatment recommendations.\n'

In [1]:
# Load the JSON knowledge base and define the forward chaining inference logic.
import json

def load_knowledge_base(filepath):
    with open(filepath, 'r') as file:
        return json.load(file)

def forward_chaining_inference(observed_symptoms, kb):
    best_match = None
    highest_score = 0

    for disease, data in kb.items():
        kb_symptoms = set(data["symptoms"])
        observed_set = set(observed_symptoms)

        match_score = len(kb_symptoms.intersection(observed_set))

        if match_score > highest_score:
            highest_score = match_score
            best_match = disease

    if best_match and highest_score > 0:
        return {
            "Diagnosis": best_match,
            "Treatments": kb[best_match]["treatment"],
            "Confidence_Score": highest_score / len(kb[best_match]["symptoms"])
        }
    return {"Diagnosis": "Unknown", "Treatments": [], "Confidence_Score": 0.0}

In [4]:
# Import TensorFlow, set image dimensions, and load the training/validation datasets from directories.
import tensorflow as tf
import matplotlib.pyplot as plt

batch_size = 32
img_height = 224
img_width = 224

train_dir = "dataset/train"
val_dir = "dataset/val"

print("Loading Training Data:")
train_ds = tf.keras.utils.image_dataset_from_directory(
  train_dir,
  seed=123,
  image_size=(img_height, img_width),
  batch_size=batch_size)

print("\nLoading Validation Data:")
val_ds = tf.keras.utils.image_dataset_from_directory(
  val_dir,
  seed=123,
  image_size=(img_height, img_width),
  batch_size=batch_size)

class_names = train_ds.class_names
print(f"\nFound {len(class_names)} disease classes.")

Loading Training Data:


FileNotFoundError: [WinError 3] The system cannot find the path specified: 'dataset/train'

In [ ]:
plt.figure(figsize=(10, 10))
for images, labels in train_ds.take(1):
    for i in range(9):
        ax = plt.subplot(3, 3, i + 1)
        plt.imshow(images[i].numpy().astype("uint8"))
        plt.title(class_names[labels[i]])
        plt.axis("off")
plt.suptitle("Sample Images from the Training Dataset")
plt.show()

In [ ]:
# Preprocessing data using pure math pixel scaling and optimize the pipeline with prefetching.
def scale_images(image, label):
    return tf.cast(image, tf.float32) / 255.0, label

train_ds = train_ds.map(scale_images)
val_ds = val_ds.map(scale_images)

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.prefetch(buffer_size=AUTOTUNE)
print("Data scaling and prefetching complete.")

In [ ]:
# Build the CNN architecture (Conv2D, MaxPooling, Dropout) and compile it with Early Stopping.
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

num_classes = len(class_names)

model = Sequential([
    Conv2D(32, (3, 3), activation='relu', input_shape=(img_height, img_width, 3)),
    MaxPooling2D(pool_size=(2, 2)),

    Conv2D(64, (3, 3), activation='relu'),
    MaxPooling2D(pool_size=(2, 2)),

    Conv2D(128, (3, 3), activation='relu'),
    MaxPooling2D(pool_size=(2, 2)),

    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.5),
    Dense(num_classes, activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True
)
print("Model architecture built and compiled successfully.")

In [ ]:
# Train the model on the dataset for up to 25 epochs and save the optimized .keras file.
print("Starting model training (up to 25 epochs, with Early Stopping)")

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=25,
    callbacks=[early_stop]
)

model.save('plant_disease_model.keras')
print("\nMODEL SAVED SUCCESSFULLY!")

In [ ]:
# Plot the training and validation accuracy/loss charts for performance evaluation.
plt.figure(figsize=(12, 4))

# Plot Accuracy
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.legend(loc='lower right')
plt.title('Training and Validation Accuracy')

# Plot Loss
plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.legend(loc='upper right')
plt.title('Training and Validation Loss')
plt.show()